# OMOP-MapAgent — End-to-End AI Agent for OMOP CDM Concept Mapping (Lab 1.5)

**Thesis context.** My thesis builds an ETL pipeline that converts three heterogeneous clinical sources
(`omop_1` Kaggle EHR, `omop_2` Synthea, `omop_3` Lithuanian kidney-biopsy registry) into the
**OMOP Common Data Model v5.4**. The hardest step is *concept mapping*: every raw source value
must become a standard OHDSI concept (SNOMED, RxNorm, LOINC, Gender, Race, Visit).

**This notebook** implements that step as one AI agent using the **Gemini API**:

| | |
|---|---|
| **Input X** | one raw source value + context: `database`, `source_table`, `source_field`, `source_value` |
| **Output y** | `target_table`, `target_field`, `concept_id`, `concept_name`, `vocabulary_id`, `domain_id`, `mapping_status` |

**Pipeline:** data understanding → prompt building → Gemini inference + reasoning → output generation & validation.

The two prompts from Lab 1.4 (`/prompts/`) are executed: **Prompt 1 (zero-shot)** and **Prompt 2 (few-shot)**.

### How to run
1. Get a key at https://aistudio.google.com/apikey.
2. In Colab click the **key icon (Secrets)** in the left sidebar → **Add new secret** → name `GEMINI_KEY`, value = your key → enable **Notebook access**.
3. **Runtime → Run all.**

## 0. Setup
Install the Google Gen AI SDK, load the API key from Colab Secrets and select a model.

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os, re, json, time
from google import genai
from google.genai import types

# --- Load the Gemini API key securely (never hardcode it) ---
try:
    from google.colab import userdata          # Colab: Tools -> Secrets -> GEMINI_KEY
    GEMINI_KEY = userdata.get('GEMINI_KEY')
except ImportError:                           # local Jupyter fallback: environment variable
    GEMINI_KEY = os.environ.get('GEMINI_KEY')

assert GEMINI_KEY, "GEMINI_KEY not found. Add it in Colab -> Secrets (key icon) and enable notebook access."
client = genai.Client(api_key=GEMINI_KEY)

# --- Choose a Gemini model ---
# Leave MODEL empty to auto-select a 'flash' model available to your key, or set it, e.g. "gemini-2.5-flash".
MODEL = ""
PREFERRED_MODELS = ["gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-2.0-flash", "gemini-flash-latest"]

def pick_model():
    try:
        names = [m.name.split("/")[-1] for m in client.models.list()
                 if "generateContent" in (getattr(m, "supported_actions", None) or [])]
    except Exception as e:
        print("Could not list models:", e)
        return PREFERRED_MODELS[0]
    for p in PREFERRED_MODELS:
        if p in names:
            return p
    flash = [n for n in names if "flash" in n and not any(w in n for w in ("image", "tts", "live", "audio"))]
    return flash[0] if flash else names[0]

MODEL = MODEL or pick_model()
print("Using model:", MODEL)

## 1. Prompts from Lab 1.4
The system prompt defines the agent's role, its four stages, the fallback rule and the JSON schema.
`USER_ZERO_SHOT` is Prompt 1; `USER_FEW_SHOT` is Prompt 2. They are recreated here (identical to the
files in `/prompts/`). Set `GITHUB_RAW_BASE` to load them from GitHub instead.

In [ ]:
# Prompts from Lab 1.4 (identical to prompts/system_prompt.txt, user_zero_shot.txt, user_few_shot.txt)
SYSTEM_PROMPT = 'You are OMOP-MapAgent, an AI agent inside a clinical ETL pipeline.\nYour job is to convert ONE raw source value (X) from a source database into the\nstandard OMOP CDM v5.4 concept (y) that the ETL writes into the target *_concept_id column.\n\nSource databases:\n- omop_1: Kaggle healthcare EHR dataset (English labels, brand-name drugs, admission types)\n- omop_2: Synthea synthetic patients (SNOMED/RxNorm/LOINC-style descriptions, encounter classes)\n- omop_3: Lithuanian kidney-biopsy registry (Lithuanian terms, abbreviations, corrupted spreadsheet values)\n\nWork through these stages:\n1. Data understanding: clean the source value (case, spaces, typos, prefixes such as "Dx:", "Rx:", "Lab:",\n   suffixes such as "(confirmed)", "daily"). Translate Lithuanian terms to English. Use source_field\n   to decide the OMOP domain (Gender, Race, Visit, Condition, Drug, Measurement).\n2. Inference: choose the standard OMOP concept. Use these vocabularies:\n   Gender/Race/Visit -> OMOP vocabularies, Condition -> SNOMED, Drug -> RxNorm ingredient, Measurement -> LOINC.\n   Map brand names to the RxNorm ingredient (for example Lipitor -> atorvastatin).\n3. Reasoning: explain briefly why this concept was chosen.\n4. Output generation: return the result in the required JSON format.\n\nRules:\n- If the value is unknown, empty, corrupted (for example "#REF!", "44562", "N/A") or a local code\n  with no standard mapping, return concept_id 0, concept_name "No matching concept",\n  vocabulary_id "None", mapping_status "unmapped".\n- Never invent a concept_id. If you are not sure about the exact id, lower the confidence.\n- Return ONLY valid JSON, no extra text.\n\nRequired JSON format:\n{\n  "cleaned_value": "<value after cleaning / translation>",\n  "reasoning": "<1-2 sentences>",\n  "target_table": "PERSON | VISIT_OCCURRENCE | CONDITION_OCCURRENCE | DRUG_EXPOSURE | MEASUREMENT",\n  "target_field": "<*_concept_id column>",\n  "concept_id": <integer>,\n  "concept_name": "<standard concept name>",\n  "vocabulary_id": "<Gender | Race | Visit | SNOMED | RxNorm | LOINC | None>",\n  "domain_id": "<Gender | Race | Visit | Condition | Drug | Measurement>",\n  "mapping_status": "mapped | unmapped",\n  "confidence": <number between 0 and 1>\n}\n'

USER_ZERO_SHOT = 'Map the following source value to a standard OMOP CDM concept.\n\nX = {x_json}\n\nReturn y as JSON in the required format.\n'

USER_FEW_SHOT = 'Map the following source value to a standard OMOP CDM concept.\nHere are solved examples from the same ETL project. Follow the same format and conventions.\n\n{examples}\n\nNow solve the new case:\nX = {x_json}\n\nReturn y as JSON in the required format.\n'

# Optional: load the prompts directly from your GitHub repository instead.
# Set GITHUB_RAW_BASE to e.g. "https://raw.githubusercontent.com/<user>/<repo>/main"
GITHUB_RAW_BASE = ""
if GITHUB_RAW_BASE:
    import urllib.request
    def _get(path):
        return urllib.request.urlopen(f"{GITHUB_RAW_BASE}/{path}").read().decode("utf-8")
    SYSTEM_PROMPT = _get("prompts/system_prompt.txt")
    USER_ZERO_SHOT = _get("prompts/user_zero_shot.txt")
    USER_FEW_SHOT = _get("prompts/user_few_shot.txt")
    print("Prompts loaded from GitHub.")

print(SYSTEM_PROMPT)

### Few-shot examples (from Lab 1.2, `data/examples.json`)
Five known (X, y) pairs: Lithuanian gender, an English condition, a Lithuanian condition,
a brand-name drug and a corrupted value (fallback to `concept_id = 0`).

In [ ]:
FEW_SHOT_EXAMPLES = [
 {
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "lytis",
   "source_value": "Moteris"
  },
  "y": {
   "target_table": "PERSON",
   "target_field": "gender_concept_id",
   "concept_id": 8532,
   "concept_name": "FEMALE",
   "vocabulary_id": "Gender",
   "domain_id": "Gender",
   "mapping_status": "mapped"
  }
 },
 {
  "X": {
   "database": "omop_1",
   "source_table": "stg_healthcare",
   "source_field": "Medical Condition",
   "source_value": "Hypertension"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 316866,
   "concept_name": "Hypertensive disorder",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "diagnoze",
   "source_value": "Lėtinė inkstų liga"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 46271022,
   "concept_name": "Chronic kidney disease",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "X": {
   "database": "omop_1",
   "source_table": "stg_healthcare",
   "source_field": "Medication",
   "source_value": "Lipitor"
  },
  "y": {
   "target_table": "DRUG_EXPOSURE",
   "target_field": "drug_concept_id",
   "concept_id": 1545958,
   "concept_name": "atorvastatin",
   "vocabulary_id": "RxNorm",
   "domain_id": "Drug",
   "mapping_status": "mapped"
  }
 },
 {
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "lytis",
   "source_value": "44562"
  },
  "y": {
   "target_table": "PERSON",
   "target_field": "gender_concept_id",
   "concept_id": 0,
   "concept_name": "No matching concept",
   "vocabulary_id": "None",
   "domain_id": "Gender",
   "mapping_status": "unmapped"
  }
 }
]

for e in FEW_SHOT_EXAMPLES:
    print(e['X']['source_value'], '->', e['y']['concept_id'], e['y']['concept_name'])

## 2. Stage 1 — Data understanding
Deterministic rules (no AI): detect the OMOP domain from the source field, clean the value
(spaces, prefixes like `Dx:`, suffixes like `(confirmed)`) and flag obviously unmappable values
(`#REF!`, `N/A`, Excel date numbers such as `44562`). The profile is used later to validate the AI output.

In [ ]:
# ============ STAGE 1: DATA UNDERSTANDING (rule-based, deterministic) ============
# Decide the OMOP domain from the source field, clean the raw value, and flag values
# that are obviously unmappable (thesis data-quality layer).

FIELD_TO_DOMAIN = {
    "gender": "Gender", "lytis": "Gender",
    "race": "Race",
    "admission type": "Visit", "encounterclass": "Visit", "apsilankymo_tipas": "Visit",
    "medical condition": "Condition", "diagnoze": "Condition",
    "medication": "Drug", "vaistai": "Drug",
    "tyrimas": "Measurement",
}
# Synthea uses the same column name DESCRIPTION in several files -> use the table name
TABLE_TO_DOMAIN = {"conditions": "Condition", "medications": "Drug", "observations": "Measurement"}

# domain -> (OMOP target table, *_concept_id column, *_source_value column)
DOMAIN_TARGET = {
    "Gender":      ("PERSON", "gender_concept_id", "gender_source_value"),
    "Race":        ("PERSON", "race_concept_id", "race_source_value"),
    "Visit":       ("VISIT_OCCURRENCE", "visit_concept_id", "visit_source_value"),
    "Condition":   ("CONDITION_OCCURRENCE", "condition_concept_id", "condition_source_value"),
    "Drug":        ("DRUG_EXPOSURE", "drug_concept_id", "drug_source_value"),
    "Measurement": ("MEASUREMENT", "measurement_concept_id", "measurement_source_value"),
}

PREFIX_RE = re.compile(r"^(history of|diagnosis|dx|dg|rx|med|tab|inj|lab|test)\s*[:.\-]?\s+", re.I)
SUFFIX_RE = re.compile(r"\s*(\((confirmed|suspected|serum|home med)\)|- chronic|daily|1x1|prn|result|value)\s*$", re.I)
GARBAGE_RE = re.compile(r"^(#[a-z/0-9]+[!?]?|n/?a|unknown|null|none|\?|-|x|\d+|ne.inoma|other|declined|see notes|see chart)$", re.I)
LOCAL_CODE_RE = re.compile(r"^[A-Z]{2,}-[A-Z0-9]+(-[A-Z0-9]+)?$")

def understand(x):
    """Return a profile of input X: domain, expected target, cleaned value, quality flags."""
    field = x["source_field"].strip().lower()
    domain = FIELD_TO_DOMAIN.get(field) or TABLE_TO_DOMAIN.get(x["source_table"].lower())
    value = re.sub(r"\s+", " ", x["source_value"]).strip()
    for _ in range(2):                       # remove up to two prefixes / suffixes
        value = PREFIX_RE.sub("", value).strip()
        value = SUFFIX_RE.sub("", value).strip()
    is_garbage = bool(GARBAGE_RE.match(value)) or value == ""
    is_local_code = bool(LOCAL_CODE_RE.match(value))
    table, field_id, field_src = DOMAIN_TARGET.get(domain, (None, None, None))
    return {
        "domain": domain, "target_table": table, "target_field": field_id,
        "source_value_field": field_src, "cleaned_value": value,
        "is_garbage": is_garbage, "is_local_code": is_local_code,
    }

understand({"database": "omop_3", "source_table": "registras", "source_field": "diagnoze",
            "source_value": "Dx: arterinė hipertenzija (confirmed)"})

## 3. Stages 2 + 3 — Prompt building, inference and reasoning
`build_prompt` fills the Lab 1.4 templates. `call_gemini` sends the system prompt + user prompt to Gemini
with `temperature = 0` (deterministic answers) and `response_mime_type = "application/json"` (machine-readable output).
The model returns `cleaned_value` and `reasoning` (its intermediate steps) together with y.

In [ ]:
# ============ STAGE 2 + 3: PROMPT BUILDING, INFERENCE AND REASONING (Gemini) ============

Y_KEYS = ["target_table", "target_field", "concept_id", "concept_name",
          "vocabulary_id", "domain_id", "mapping_status"]

def format_examples(examples, with_distance=False):
    parts = []
    for i, e in enumerate(examples, 1):
        y = {k: e["y"][k] for k in Y_KEYS}
        head = f"Example {i}" + (f" (similarity distance {e['distance']:.3f})" if with_distance else "")
        parts.append(f"{head}\nX = {json.dumps(e['X'], ensure_ascii=False)}\ny = {json.dumps(y, ensure_ascii=False)}")
    return "\n\n".join(parts)

def build_prompt(x, examples=None, with_distance=False):
    x_json = json.dumps(x, ensure_ascii=False)
    if not examples:
        return USER_ZERO_SHOT.format(x_json=x_json)
    return USER_FEW_SHOT.format(examples=format_examples(examples, with_distance), x_json=x_json)

PAUSE_SECONDS = 4      # spacing between calls to respect free-tier rate limits

def call_gemini(user_prompt, retries=4):
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=0.0,
        response_mime_type="application/json",
    )
    for attempt in range(retries):
        try:
            response = client.models.generate_content(model=MODEL, contents=user_prompt, config=config)
            time.sleep(PAUSE_SECONDS)
            return response.text or ""
        except Exception as e:
            wait = 20 * (attempt + 1)
            print(f"   API error ({type(e).__name__}: {str(e)[:150]}) -> retry in {wait}s")
            time.sleep(wait)
    return ""

## 4. Stage 4 — Output generation and validation
The AI answer is never written to the database blindly. The validator checks the JSON, forces the target table
to match the source field, and applies the thesis fallback rule. Finally it builds the OMOP row fragment
(`*_concept_id` + `*_source_value`).

In [ ]:
# ============ STAGE 4: OUTPUT GENERATION + VALIDATION ============
# Parse the model answer, check it against the stage-1 profile, enforce the thesis fallback rule
# (unmappable -> concept_id 0, raw text kept in *_source_value) and build the OMOP row.

def validate(raw_text, x, profile):
    warnings = []
    try:
        y = json.loads(raw_text)
        if isinstance(y, list):
            y = y[0]
    except Exception:
        y = {}
        warnings.append("model output is not valid JSON")
    try:
        y["concept_id"] = int(y.get("concept_id", 0))
    except (TypeError, ValueError):
        y["concept_id"] = 0
        warnings.append("concept_id was not an integer")

    # Domain / target table must agree with the source field (deterministic routing wins)
    if profile["domain"]:
        if y.get("domain_id") != profile["domain"] and y["concept_id"] != 0:
            warnings.append(f"model domain {y.get('domain_id')} != expected {profile['domain']}")
        y["domain_id"] = profile["domain"]
        y["target_table"] = profile["target_table"]
        y["target_field"] = profile["target_field"]

    # Fallback rule
    if profile["is_garbage"] and y["concept_id"] != 0:
        warnings.append("value is unmappable by rule -> concept_id forced to 0")
        y["concept_id"] = 0
    if y["concept_id"] == 0:
        y.update(concept_name="No matching concept", vocabulary_id="None", mapping_status="unmapped")
    else:
        y["mapping_status"] = "mapped"

    omop_row = {y.get("target_field") or "concept_id": y["concept_id"],
                profile["source_value_field"] or "source_value": x["source_value"]}
    return y, omop_row, warnings

In [ ]:
# ============ THE END-TO-END AGENT ============

def run_agent(x, examples=None, with_distance=False, verbose=True):
    profile = understand(x)                                  # Stage 1
    prompt = build_prompt(x, examples, with_distance)        # Stage 2
    raw = call_gemini(prompt)                                # Stage 3
    y, row, warnings = validate(raw, x, profile)             # Stage 4
    if verbose:
        line = "=" * 90
        print(line, "\nINPUT X:\n", json.dumps(x, ensure_ascii=False, indent=2))
        print(line, "\nSTAGE 1 - data understanding:\n", json.dumps(profile, ensure_ascii=False, indent=2))
        print(line, "\nSTAGE 2 - prompt sent to Gemini (user part):\n", prompt)
        print(line, "\nSTAGE 3 - raw Gemini answer (reasoning + inference):\n", raw)
        print(line, "\nSTAGE 4 - validated output y:\n", json.dumps(y, ensure_ascii=False, indent=2))
        print("OMOP row fragment written by the ETL:", row)
        if warnings:
            print("Validation warnings:", warnings)
    return y

## 5. Demonstration — Prompt 1 (zero-shot)
A new observation from the Lithuanian registry: `"Dx: arterinė hipertenzija (confirmed)"` (arterial hypertension).
No examples are given to the model.

In [ ]:
X_NEW = {"database": "omop_3", "source_table": "registras", "source_field": "diagnoze",
         "source_value": "Dx: arterinė hipertenzija (confirmed)"}

y_zero = run_agent(X_NEW)

## 6. Demonstration — Prompt 2 (few-shot)
Same X, but the five Lab 1.2 examples are included in the prompt.

In [ ]:
y_few = run_agent(X_NEW, examples=FEW_SHOT_EXAMPLES)

### More end-to-end runs (one per source database and domain)

In [ ]:
more_inputs = [
    {"database": "omop_1", "source_table": "stg_healthcare", "source_field": "Medication", "source_value": "Ibuprofen"},
    {"database": "omop_2", "source_table": "observations", "source_field": "DESCRIPTION", "source_value": "Hemoglobin A1c/Hemoglobin.total in Blood"},
    {"database": "omop_3", "source_table": "registras", "source_field": "lytis", "source_value": "#REF!"},
]
for x in more_inputs:
    run_agent(x, examples=FEW_SHOT_EXAMPLES)

## 7. Visualization — process flow of the agent

In [ ]:
import graphviz

def show_flow(stages, title):
    g = graphviz.Digraph(graph_attr={"rankdir": "LR", "label": title, "labelloc": "t"},
                         node_attr={"shape": "box", "style": "rounded,filled", "fontname": "Helvetica"})
    colors = {"io": "#dbeafe", "rule": "#e5e7eb", "ai": "#fde68a", "db": "#dcfce7"}
    for name, kind in stages:
        g.node(name, fillcolor=colors[kind])
    for (a, _), (b, _) in zip(stages, stages[1:]):
        g.edge(a, b)
    return g

show_flow([("Input X\n(raw source value)", "io"),
           ("Stage 1\nData understanding\n(rules)", "rule"),
           ("Stage 2\nPrompt building\n(zero / few-shot)", "rule"),
           ("Stage 3\nGemini inference\n+ reasoning", "ai"),
           ("Stage 4\nValidation +\nfallback rule", "rule"),
           ("Output y\n(OMOP concept)", "io")],
          "OMOP-MapAgent end-to-end pipeline")

## 8. Evaluation — zero-shot vs few-shot on the held-out test set
`TEST_SET` contains 40 noisy source values (typos, Lithuanian, prefixes, corrupted values) with known y
(`data/test_set.json`). `N_EVAL` controls how many are used (each case = 2 API calls).
Metric: **exact `concept_id` match** with the ground truth.

In [ ]:
TEST_SET = [
 {
  "id": "test_001",
  "X": {
   "database": "omop_1",
   "source_table": "stg_healthcare",
   "source_field": "Medication",
   "source_value": "Tab. #N/A"
  },
  "y": {
   "target_table": "DRUG_EXPOSURE",
   "target_field": "drug_concept_id",
   "concept_id": 0,
   "concept_name": "No matching concept",
   "vocabulary_id": "None",
   "domain_id": "Drug",
   "mapping_status": "unmapped"
  }
 },
 {
  "id": "test_002",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "lytis",
   "source_value": "Vyas"
  },
  "y": {
   "target_table": "PERSON",
   "target_field": "gender_concept_id",
   "concept_id": 8507,
   "concept_name": "MALE",
   "vocabulary_id": "Gender",
   "domain_id": "Gender",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_003",
  "X": {
   "database": "omop_2",
   "source_table": "observations",
   "source_field": "DESCRIPTION",
   "source_value": "serum albumin"
  },
  "y": {
   "target_table": "MEASUREMENT",
   "target_field": "measurement_concept_id",
   "concept_id": 3024561,
   "concept_name": "Albumin [Mass/volume] in Serum or Plasma",
   "vocabulary_id": "LOINC",
   "domain_id": "Measurement",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_004",
  "X": {
   "database": "omop_1",
   "source_table": "stg_healthcare",
   "source_field": "Medical Condition",
   "source_value": "Dx: asthma"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 317009,
   "concept_name": "Asthma",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_005",
  "X": {
   "database": "omop_2",
   "source_table": "observations",
   "source_field": "DESCRIPTION",
   "source_value": "Body Weight value"
  },
  "y": {
   "target_table": "MEASUREMENT",
   "target_field": "measurement_concept_id",
   "concept_id": 3025315,
   "concept_name": "Body weight",
   "vocabulary_id": "LOINC",
   "domain_id": "Measurement",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_006",
  "X": {
   "database": "omop_2",
   "source_table": "medications",
   "source_field": "DESCRIPTION",
   "source_value": "Rx: Furoseimde 40 MG Oral Tablet"
  },
  "y": {
   "target_table": "DRUG_EXPOSURE",
   "target_field": "drug_concept_id",
   "concept_id": 956874,
   "concept_name": "furosemide",
   "vocabulary_id": "RxNorm",
   "domain_id": "Drug",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_007",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "lytis",
   "source_value": "Mooteris"
  },
  "y": {
   "target_table": "PERSON",
   "target_field": "gender_concept_id",
   "concept_id": 8532,
   "concept_name": "FEMALE",
   "vocabulary_id": "Gender",
   "domain_id": "Gender",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_008",
  "X": {
   "database": "omop_1",
   "source_table": "stg_healthcare",
   "source_field": "Medical Condition",
   "source_value": "history of kita"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 0,
   "concept_name": "No matching concept",
   "vocabulary_id": "None",
   "domain_id": "Condition",
   "mapping_status": "unmapped"
  }
 },
 {
  "id": "test_009",
  "X": {
   "database": "omop_2",
   "source_table": "patients",
   "source_field": "RACE",
   "source_value": "African  American"
  },
  "y": {
   "target_table": "PERSON",
   "target_field": "race_concept_id",
   "concept_id": 8516,
   "concept_name": "Black or African American",
   "vocabulary_id": "Race",
   "domain_id": "Race",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_010",
  "X": {
   "database": "omop_2",
   "source_table": "observations",
   "source_field": "DESCRIPTION",
   "source_value": "LAB - k+"
  },
  "y": {
   "target_table": "MEASUREMENT",
   "target_field": "measurement_concept_id",
   "concept_id": 3023103,
   "concept_name": "Potassium [Moles/volume] in Serum or Plasma",
   "vocabulary_id": "LOINC",
   "domain_id": "Measurement",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_011",
  "X": {
   "database": "omop_2",
   "source_table": "observations",
   "source_field": "DESCRIPTION",
   "source_value": "LAB - glucose [mass/volume] in serum or plasma"
  },
  "y": {
   "target_table": "MEASUREMENT",
   "target_field": "measurement_concept_id",
   "concept_id": 3004501,
   "concept_name": "Glucose [Mass/volume] in Serum or Plasma",
   "vocabulary_id": "LOINC",
   "domain_id": "Measurement",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_012",
  "X": {
   "database": "omop_2",
   "source_table": "conditions",
   "source_field": "DESCRIPTION",
   "source_value": "CHRONIC KIDNEY DISEASE (DISORDER) - chronic"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 46271022,
   "concept_name": "Chronic kidney disease",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_013",
  "X": {
   "database": "omop_2",
   "source_table": "medications",
   "source_field": "DESCRIPTION",
   "source_value": "inj. Prednisone   "
  },
  "y": {
   "target_table": "DRUG_EXPOSURE",
   "target_field": "drug_concept_id",
   "concept_id": 1551099,
   "concept_name": "prednisone",
   "vocabulary_id": "RxNorm",
   "domain_id": "Drug",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_014",
  "X": {
   "database": "omop_2",
   "source_table": "conditions",
   "source_field": "DESCRIPTION",
   "source_value": "OSTEOARRTHRITIS (DISORDER)"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 80180,
   "concept_name": "Osteoarthritis",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_015",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "diagnoze",
   "source_value": "CD   "
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 201820,
   "concept_name": "Diabetes mellitus",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_016",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "diagnoze",
   "source_value": "history of CD"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 201820,
   "concept_name": "Diabetes mellitus",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_017",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "tyrimas",
   "source_value": "N/A (serum)"
  },
  "y": {
   "target_table": "MEASUREMENT",
   "target_field": "measurement_concept_id",
   "concept_id": 0,
   "concept_name": "No matching concept",
   "vocabulary_id": "None",
   "domain_id": "Measurement",
   "mapping_status": "unmapped"
  }
 },
 {
  "id": "test_018",
  "X": {
   "database": "omop_2",
   "source_table": "encounters",
   "source_field": "ENCOUNTERCLASS",
   "source_value": "  ambulatory"
  },
  "y": {
   "target_table": "VISIT_OCCURRENCE",
   "target_field": "visit_concept_id",
   "concept_id": 9202,
   "concept_name": "Outpatient Visit",
   "vocabulary_id": "Visit",
   "domain_id": "Visit",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_019",
  "X": {
   "database": "omop_2",
   "source_table": "conditions",
   "source_field": "DESCRIPTION",
   "source_value": "DX: T2DM"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 201826,
   "concept_name": "Type 2 diabetes mellitus",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_020",
  "X": {
   "database": "omop_2",
   "source_table": "medications",
   "source_field": "DESCRIPTION",
   "source_value": "  Omeprazole 20 MG Delayed Release Oral Capsule"
  },
  "y": {
   "target_table": "DRUG_EXPOSURE",
   "target_field": "drug_concept_id",
   "concept_id": 923645,
   "concept_name": "omeprazole",
   "vocabulary_id": "RxNorm",
   "domain_id": "Drug",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_021",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "apsilankymo_tipas",
   "source_value": "AMBULATORINĖ  KONSULTACIJA"
  },
  "y": {
   "target_table": "VISIT_OCCURRENCE",
   "target_field": "visit_concept_id",
   "concept_id": 9202,
   "concept_name": "Outpatient Visit",
   "vocabulary_id": "Visit",
   "domain_id": "Visit",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_022",
  "X": {
   "database": "omop_2",
   "source_table": "medications",
   "source_field": "DESCRIPTION",
   "source_value": "Norvsc"
  },
  "y": {
   "target_table": "DRUG_EXPOSURE",
   "target_field": "drug_concept_id",
   "concept_id": 1332418,
   "concept_name": "amlodipine",
   "vocabulary_id": "RxNorm",
   "domain_id": "Drug",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_023",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "diagnoze",
   "source_value": "history of plaučių uždegimas (suspected)"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 255848,
   "concept_name": "Pneumonia",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_024",
  "X": {
   "database": "omop_2",
   "source_table": "medications",
   "source_field": "DESCRIPTION",
   "source_value": "Norvasc daily"
  },
  "y": {
   "target_table": "DRUG_EXPOSURE",
   "target_field": "drug_concept_id",
   "concept_id": 1332418,
   "concept_name": "amlodipine",
   "vocabulary_id": "RxNorm",
   "domain_id": "Drug",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_025",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "tyrimas",
   "source_value": "  Hemoglobinas"
  },
  "y": {
   "target_table": "MEASUREMENT",
   "target_field": "measurement_concept_id",
   "concept_id": 3000963,
   "concept_name": "Hemoglobin [Mass/volume] in Blood",
   "vocabulary_id": "LOINC",
   "domain_id": "Measurement",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_026",
  "X": {
   "database": "omop_2",
   "source_table": "medications",
   "source_field": "DESCRIPTION",
   "source_value": "24 HR Metfomin 500 MG Extended Release Oral Tablet"
  },
  "y": {
   "target_table": "DRUG_EXPOSURE",
   "target_field": "drug_concept_id",
   "concept_id": 1503297,
   "concept_name": "metformin",
   "vocabulary_id": "RxNorm",
   "domain_id": "Drug",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_027",
  "X": {
   "database": "omop_2",
   "source_table": "conditions",
   "source_field": "DESCRIPTION",
   "source_value": "pulmonary emphyssema (disorder)"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 255573,
   "concept_name": "Chronic obstructive lung disease",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_028",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "apsilankymo_tipas",
   "source_value": "Ambulaorinis"
  },
  "y": {
   "target_table": "VISIT_OCCURRENCE",
   "target_field": "visit_concept_id",
   "concept_id": 9202,
   "concept_name": "Outpatient Visit",
   "vocabulary_id": "Visit",
   "domain_id": "Visit",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_029",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "tyrimas",
   "source_value": "test: gilukozė kraujyje"
  },
  "y": {
   "target_table": "MEASUREMENT",
   "target_field": "measurement_concept_id",
   "concept_id": 3004501,
   "concept_name": "Glucose [Mass/volume] in Serum or Plasma",
   "vocabulary_id": "LOINC",
   "domain_id": "Measurement",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_030",
  "X": {
   "database": "omop_2",
   "source_table": "conditions",
   "source_field": "DESCRIPTION",
   "source_value": "dx - Diaabetes"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 201820,
   "concept_name": "Diabetes mellitus",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_031",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "lytis",
   "source_value": "Moetris"
  },
  "y": {
   "target_table": "PERSON",
   "target_field": "gender_concept_id",
   "concept_id": 8532,
   "concept_name": "FEMALE",
   "vocabulary_id": "Gender",
   "domain_id": "Gender",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_032",
  "X": {
   "database": "omop_2",
   "source_table": "encounters",
   "source_field": "ENCOUNTERCLASS",
   "source_value": "inpatient   "
  },
  "y": {
   "target_table": "VISIT_OCCURRENCE",
   "target_field": "visit_concept_id",
   "concept_id": 9201,
   "concept_name": "Inpatient Visit",
   "vocabulary_id": "Visit",
   "domain_id": "Visit",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_033",
  "X": {
   "database": "omop_2",
   "source_table": "conditions",
   "source_field": "DESCRIPTION",
   "source_value": "Dg. Atrial fibrillation (disorder)"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 313217,
   "concept_name": "Atrial fibrillation",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_034",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "diagnoze",
   "source_value": "Miokardo Infarktas (Confirmed)"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 4329847,
   "concept_name": "Myocardial infarction",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_035",
  "X": {
   "database": "omop_2",
   "source_table": "conditions",
   "source_field": "DESCRIPTION",
   "source_value": "Major deepression disorder (disorder)"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 440383,
   "concept_name": "Depressive disorder",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_036",
  "X": {
   "database": "omop_2",
   "source_table": "conditions",
   "source_field": "DESCRIPTION",
   "source_value": "Myocardial infarction (disorder) (suspected)"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 4329847,
   "concept_name": "Myocardial infarction",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_037",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "diagnoze",
   "source_value": "cukrinis diabetas  "
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 201820,
   "concept_name": "Diabetes mellitus",
   "vocabulary_id": "SNOMED",
   "domain_id": "Condition",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_038",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "tyrimas",
   "source_value": "test: Albuminas"
  },
  "y": {
   "target_table": "MEASUREMENT",
   "target_field": "measurement_concept_id",
   "concept_id": 3024561,
   "concept_name": "Albumin [Mass/volume] in Serum or Plasma",
   "vocabulary_id": "LOINC",
   "domain_id": "Measurement",
   "mapping_status": "mapped"
  }
 },
 {
  "id": "test_039",
  "X": {
   "database": "omop_3",
   "source_table": "registras",
   "source_field": "diagnoze",
   "source_value": "  #N/A"
  },
  "y": {
   "target_table": "CONDITION_OCCURRENCE",
   "target_field": "condition_concept_id",
   "concept_id": 0,
   "concept_name": "No matching concept",
   "vocabulary_id": "None",
   "domain_id": "Condition",
   "mapping_status": "unmapped"
  }
 },
 {
  "id": "test_040",
  "X": {
   "database": "omop_2",
   "source_table": "medications",
   "source_field": "DESCRIPTION",
   "source_value": "  Glucophage"
  },
  "y": {
   "target_table": "DRUG_EXPOSURE",
   "target_field": "drug_concept_id",
   "concept_id": 1503297,
   "concept_name": "metformin",
   "vocabulary_id": "RxNorm",
   "domain_id": "Drug",
   "mapping_status": "mapped"
  }
 }
]
print(len(TEST_SET), 'test examples')

In [ ]:
import pandas as pd

N_EVAL = 20
rows = []
for t in TEST_SET[:N_EVAL]:
    for mode, ex in [("zero-shot", None), ("few-shot", FEW_SHOT_EXAMPLES)]:
        y = run_agent(t["X"], examples=ex, verbose=False)
        rows.append({"id": t["id"], "mode": mode, "source_value": t["X"]["source_value"],
                     "true_id": t["y"]["concept_id"], "true_name": t["y"]["concept_name"],
                     "pred_id": y["concept_id"], "pred_name": y.get("concept_name"),
                     "correct": y["concept_id"] == t["y"]["concept_id"]})
    print(f"{t['id']} done")

results = pd.DataFrame(rows)
summary = results.groupby("mode")["correct"].mean().rename("accuracy")
print(summary)

In [ ]:
import matplotlib.pyplot as plt

ax = summary.reindex(["zero-shot", "few-shot"]).plot(kind="bar", ylim=(0, 1), rot=0,
                                                     color=["#94a3b8", "#2563eb"],
                                                     title="Exact concept_id accuracy")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.0%}", (p.get_x() + p.get_width() / 2, p.get_height() + 0.02), ha="center")
plt.ylabel("accuracy"); plt.show()

print("Wrong answers:")
results[~results["correct"]][["mode", "source_value", "true_id", "true_name", "pred_id", "pred_name"]]

## 9. Reflection

*(Update the numbers in brackets with the accuracy printed in section 8.)*

OMOP-MapAgent works as a complete AI agent for the concept-mapping step of my OMOP CDM ETL thesis: it receives one raw
source value X from any of the three source databases and returns the standard OMOP concept y that the ETL writes into
the `*_concept_id` column. The pipeline combines deterministic rules (domain routing, cleaning, the `concept_id = 0`
fallback) with Gemini, which does the language-heavy part: understanding typos, abbreviations, brand names and
Lithuanian clinical terms. Prompt engineering defines the agent: the system prompt fixes the role, the four stages, the
fallback rule and a strict JSON schema, and `response_mime_type="application/json"` with temperature 0 makes the output
machine-readable and repeatable. Zero-shot prompting reached [__ %] exact-id accuracy and few-shot prompting [__ %].
What worked well: the domain, target table and concept name were almost always right, Lithuanian values were translated
correctly, and the validator caught corrupted inputs. The weak point is the exact numeric `concept_id`: the model must
recall OHDSI ids from memory, so it sometimes picks a related concept or a wrong number when the examples do not contain
the concept. The five few-shot examples are fixed, so they help only for similar values. An improvement is to choose the
examples dynamically from a large database of known mappings, which is implemented in Lab 1.6 (RAG with ChromaDB);
a second improvement is to verify every returned id against the ATHENA vocabulary tables.